## Testing Decoder-only Transformer Model

In [45]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[2] # cwd up two directories
sys.path.append(str(ROOT))

In [46]:
import torch

from src.llm.config import ModelConfig
from src.llm.model import DecoderTransformer

In [47]:


config = ModelConfig(
    vocab_size=1000,
    d_model=8,
    num_heads=2,
    d_ff=32,
    num_layers=3
)

model = DecoderTransformer(config)

token_ids = torch.randint(
    0,
    config.vocab_size,
    (2, 4)
)

logits = model(token_ids)

print("Token IDs shape:", token_ids.shape)
print("Logits shape:", logits.shape)

Token IDs shape: torch.Size([2, 4])
Logits shape: torch.Size([2, 4, 1000])


### Sanity Checks

Check LM head output tensor.

In [48]:
assert logits.shape == (2, 4, config.vocab_size)

print("Logits shape corresponds to model configs!")

Logits shape corresponds to model configs!


Check weight tying.

In [49]:
assert model.token_embedding.weight is model.lm_head.weight

print("Embeddings and output projection weights are the same!")

embedding_params = model.token_embedding.weight.numel()
lm_head_params = model.lm_head.weight.numel()

print("\nEmbedding parameters:", embedding_params)
print("LM head parameters:", lm_head_params)

Embeddings and output projection weights are the same!

Embedding parameters: 8000
LM head parameters: 8000


Sum up total parameters.

In [50]:
total_params = sum(parameter.numel() for parameter in model.parameters())

print(f"Total parameters: {total_params:,}")

Total parameters: 11,128


## Backprop Check

In [51]:
import torch.nn.functional as F

token_ids = torch.randint(
    0,
    config.vocab_size,
    (2, 8)
)

# Shift tokens for next-token prediction
inputs = token_ids[:, :-1] # drop last
targets = token_ids[:, 1:] # drop first

print("Inputs:", inputs.shape)
print("Targets:", targets.shape)

logits = model(inputs)

print("\nLogits:", logits.shape)

Inputs: torch.Size([2, 7])
Targets: torch.Size([2, 7])

Logits: torch.Size([2, 7, 1000])


### Cross-entropy

Cross-entropy compares its $V$ vocabulary logits against the **actual/correct next-token ID** and averages the error across all tokens in the batch.

For cross-entropy, batches aren't particularly important and only **each independent token predictions** are looked at.

We reshape because cross-entropy expects a list of **individual classification predictions**.

---

Batch and sequence dimensions get flattened so each token position is treated as an individual prediction: \
$[B,T,V] \rightarrow [B \times T, V]$ or $[Z,V]$

Conceptually for an **logit output tensor** with batch and token positions $[2,3]$: \
$
\begin{array}{c|ccc}
 & T_1 & T_2 & T_3 \\ \hline
B_1 & \mathbf{z}_{1,1} & \mathbf{z}_{1,2} & \mathbf{z}_{1,3} \\
B_2 & \mathbf{z}_{2,1} & \mathbf{z}_{2,2} & \mathbf{z}_{2,3}
\end{array}
\longrightarrow
\begin{bmatrix}
\mathbf{z}_{1,1} \\
\mathbf{z}_{1,2} \\
\mathbf{z}_{1,3} \\
\mathbf{z}_{2,1} \\
\mathbf{z}_{2,2} \\
\mathbf{z}_{2,3}
\end{bmatrix}
$, 
Each $\mathbf{z}_{b,t}$ is a **$V$-dimensional** vector for one next-token prediction. \
Reshaping simply unwraps the $B \times T$ grid of token positions into $BT$ rows, predictions (logits) stay unchanged.

**Targets**: \
$
\begin{bmatrix}
y_{1,1} & y_{1,2} & y_{1,3} \\
y_{2,1} & y_{2,2} & y_{2,3}
\end{bmatrix}
\quad\longrightarrow\quad
\begin{bmatrix}
y_{1,1} \\
y_{1,2} \\
y_{1,3} \\
y_{2,1} \\
y_{2,2} \\
y_{2,3}
\end{bmatrix}
$,
Each row of $V$ logits lines up with exactly one correct next-token ID. \
logits $[BT,V] \leftrightarrow [BT]$ correct/target token IDS.

https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.cross_entropy.html

In [52]:
loss = F.cross_entropy(
    logits.reshape(-1, config.vocab_size), # [B*T, V]
    targets.reshape(-1) # [B*T]
)

print("Loss:", loss.item())

Loss: 9.856504440307617


### Backpropagation

Cross-entropy calculated how **wrong** the model was, backpropagation determines which parameters contributed to the **loss score**.

Conceptually `backward()` traces backward through the **computations that produced the loss** and calculates the **gradient** of the loss with respect to each trainable parameter.

---

PyTorch stores the computed gradients in each parameter's `.grad`, where the optimizer can later use them to update the model.
It's important to note that backpropagation **does not update the weights**. It only **calculates** and **stores** their gradients.

In [ ]:
loss.backward()

# Check some parameters for gradients
print("Embedding gradient:", model.token_embedding.weight.grad is not None)

print("Attention gradient:", model.blocks[0].attn.qkv_proj.weight.grad is not None)

print("MLP gradient:", model.blocks[0].mlp.up_proj.weight.grad is not None)

print("RMSNorm gradient:", model.blocks[0].attn_norm.weight.grad is not None)

Embedding gradient: True
Attention gradient: True
MLP gradient: True
